# 02 - Executing prompts programmatically

## Running prompts programmatically

In [4]:
from openai import OpenAI

In [5]:
client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)

In [6]:
prompt_input = """Write a concise message to remind
users to be vigilant about phishing attacks."""

response = client.chat.completions.create(
    model="granite4.1:3b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt_input}
    ]
)

print(response)

ChatCompletion(id='chatcmpl-453', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Reminder: Stay alert for phishing attempts. Verify sender's identity, check email URLs carefully, and avoid sharing personal information via unsolicited messages. Your vigilance protects your data.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1782009463, model='granite4.1:3b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=36, prompt_tokens=34, total_tokens=70, completion_tokens_details=None, prompt_tokens_details=None))


In [7]:
# Explore the attributes and properties of the response object
print(response.choices[0].message.content)

Reminder: Stay alert for phishing attempts. Verify sender's identity, check email URLs carefully, and avoid sharing personal information via unsolicited messages. Your vigilance protects your data.


## Running prompts with LangChain

In [8]:
from langchain_ollama import ChatOllama

In [9]:
llm = ChatOllama(model="granite4.1:3b")

In [10]:
prompt_input = """Write a concise message to remind 
users to be vigilant about phishing attacks."""

In [11]:
response = llm.invoke(prompt_input)
print(response.content)

Stay alert: Always verify the sender’s identity and look for secure URLs before clicking links or entering sensitive information. Phishing attempts can appear convincing; if in doubt, contact the organization directly via their official website or known contact details. Your security matters!


## Prompt templates

### Implementing a prompt template with a Python function

In [13]:
def generate_text_summary_prompt(text, num_words, tone):
    return f"You are an experienced copywriter. Write a {num_words} words summary of the following text, using a {tone} tone: {text}"

In [14]:
segovia_aqueduct_text = """The Aqueduct of Segovia (Spanish:
Acueducto de Segovia) is a Roman aqueduct in Segovia, Spain.
It was built around the first century AD to channel water from
springs in the mountains 17 kilometres (11 mi) away to the
city's fountains, public baths and private houses, and was in
use until 1973.
Its elevated section, with its complete arcade of 167 arches,
is one of the best-preserved Roman aqueduct bridges and the
foremost symbol of Segovia, as evidenced by its presence on the
city's coat of arms.
The Old Town of Segovia and the aqueduct, were declared a UNESCO
World Heritage Site in 1985. As the aqueduct lacks a legible
inscription (one was apparently located in the structure's attic,
or top portion[citation needed]), the date of construction cannot be
definitively determined. The general date of the Aqueduct's
construction was long a mystery, although it was thought to have
been during the 1st century AD, during the reigns of the Emperors
Domitian, Nerva, and Trajan. At the end of the 20th century,
Géza Alföldy deciphered the text on the dedication plaque by
studying the anchors that held the now missing bronze letters
in place. He determined that Emperor Domitian (AD 81–96) ordered
its construction[1] and the year 98 AD was proposed as the most
likely date of completion.[2] However, in 2016 archeological
evidence was published which points to a slightly later date,
after 112 AD, during the government of Trajan or in the
beginning of the government of emperor Hadrian, from 117 AD."""

In [15]:
input_prompt = generate_text_summary_prompt(
    text=segovia_aqueduct_text,
    num_words=20,
    tone="knowledgeable and engaging"
)

In [16]:
response = llm.invoke(input_prompt)
print(response.content)

The Aqueduct of Segovia, a Roman marvel built around 1st century AD, supplies water to Segovia's fountains and baths till 1973; its preserved arches symbolize the city and are UNESCO World Heritage Sites since 1985.


### Using LangChain's PromptTemplate

With LangChain, you don't need to implement a prompt template function manually. Instead, you can use the convenient _PromptTemplate_ class to handle parametrized templates.

In [17]:
from langchain_core.prompts import PromptTemplate

In [18]:
prompt_template = PromptTemplate.from_template(
""" You are an experienced copywriter.
Write a {num_words} words summary of the following text,
using a {tone} tone: {text}
""")

To use the prompt template, create a prompt instance, and format it with your parameters:

In [19]:
prompt = prompt_template.format(
    text=segovia_aqueduct_text,
    num_words=20,
    tone="knowledgeable and engaging"
)

Then, invoke the Chat client with the formatted prompt:

In [20]:
response = llm.invoke(prompt)
print(response.content)

The Aqueduct of Segovia, a Roman marvel built ~98 AD under Domitian, spans 17 km to supply water to ancient Spain until 1973. Its iconic elevated arches, now UNESCO-listed, remain a symbol of Segovia's rich history.


## Few-shot learning

In [30]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="granite4.1:3b")

In [24]:
prompt_input = """
Classify the following numbers as Abra, Kadabra or Kadabra:
3, 4, 5, 7, 8, 10, 11, 13, 35
Examples:
6 // not divisible by 5, not divisible by 7 // None
15 // divisible by 5, not divisible by 7 // Abra
12 // not divisible by 5, not divisible by 7 // None
21 // not divisible by 5, divisible by 7 // Kadabra
70 // divisible by 5, divisible by 7 // Abra Kadabra
"""

In [31]:
response = llm.invoke(prompt_input)

print(response.content)

Here's the classification for the given numbers based on the examples provided:

3 // not divisible by 5, not divisible by 7 // None  
4 // not divisible by 5, not divisible by 7 // None  
5 // divisible by 5, not divisible by 7 // Abra  
7 // not divisible by 5, divisible by 7 // Kadabra  
8 // not divisible by 5, not divisible by 7 // None  
10 // divisible by 5, not divisible by 7 // Abra  
11 // not divisible by 5, not divisible by 7 // None  
13 // not divisible by 5, not divisible by 7 // None  
35 // divisible by 5, divisible by 7 // Abra Kadabra


### Using Langchain's FewShotPromptTemplate

In [32]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [33]:
examples = [
    {
        "number": 6,
        "reasoning": "not divisible by 5 nor by 7",
        "result": "None"
    },
    {
        "number": 15,
        "reasoning": "divisible by 5 but not by 7",
        "result": "Abra"
    },
    {
        "number": 12,
        "reasoning": "not divisible by 5 nor by 7",
        "result": "None"
    },
    {
        "number": 21,
        "reasoning": "divisible by 7 but not by 5",
        "result": "Kadabra"
    },
    {
        "number": 70,
        "reasoning": "divisible by 5 and by 7",
        "result": "Abra Kadabra"
    }
]

In [34]:
example_prompt = PromptTemplate(
    input_variables=["number", "reasoning", "result"],
    template="{number}  \\ {reasoning} \\ {result}"
)

In [35]:
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Classify the following numbers as Abra, Kadabra or Abra Kadabra: {comma_delimited_input_numbers}",
    input_variables=["comma_delimited_input_numbers"]
)

In [36]:
prompt_input = few_shot_prompt.format(
    comma_delimited_input_numbers="3, 4, 5, 7, 8, 10, 11, 13, 35."
)

In [37]:
response = llm.invoke(prompt_input)
print(response.content)

To classify the given numbers (3, 4, 5, 7, 8, 10, 11, 13, 35) as Abra, Kadabra, or Abra Kadabra, we need to determine their divisibility by 5 and/or 7:

- **Abra**: Divisible by 5 but not by 7.
- **Kadabra**: Divisible by 7 but not by 5.
- **Abra Kadabra**: Divisible by both 5 and 7.

Now let's classify each number:

1. **3**: Not divisible by 5 or 7 → None
2. **4**: Not divisible by 5 or 7 → None
3. **5**: Divisible by 5 but not by 7 → Abra
4. **7**: Divisible by 7 but not by 5 → Kadabra
5. **8**: Not divisible by 5 or 7 → None
6. **10**: Divisible by 5 but not by 7 → Abra
7. **11**: Not divisible by 5 or 7 → None
8. **13**: Not divisible by 5 or 7 → None
9. **35**: Divisible by both 5 and 7 → Abra Kadabra

So the classification is:

- 3: None
- 4: None
- 5: Abra
- 7: Kadabra
- 8: None
- 10: Abra
- 11: None
- 13: None
- 35: Abra Kadabra


## Chain of Thought
The Chain of Thought (CoT) technique blends providing logical steps with few-shot learning.

A sequence of numbers is called "strange" if it contains at least two odd numbers and the sum of all odd numbers is divisible by 3.

In [41]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="granite4.1:3b")

In [42]:
prompt_input = """
Q: Is the following sequence “strange”?
3, 4, 5, 7, 10, 18, 22, 24
Examples:
Q: is the following sequence strange: 1, 4, 6, 8, 20
A: 1 is an odd number; I need at least two odd numbers // Not Strange
Q: is the following sequence strange: 5, 6, 7, 8, 20
A: 5 and 7 are odd numbers; the sum of 5 and 7 is 12; 12 is divisible by 3 // Strange
Q: is the following sequence strange: 1, 5, 6, 7, 8, 20
A: 1, 5 and 7 are odd numbers; the sum of 1, 5 and 7 is 13; 13 is not divisible by 3 // Not Strange
Q: is the following sequence strange: 5, 6, 7, 8, 9, 20
A: 5, 7, 9 are odd numbers; the sum of 5, 7 and 9 is 21; 21 is divisible by 3 // Strange
"""

In [43]:
response = llm.invoke(prompt_input)
print(response.content)

To determine if the sequence “3, 4, 5, 7, 10, 18, 22, 24” is strange, we need to apply a specific rule or pattern that defines what makes a sequence “strange.” From the examples provided, it appears that a sequence is considered “Strange” if there are at least two odd numbers whose sum is divisible by 3.

Let's analyze the given sequence:

1. Identify the odd numbers in the sequence: 3, 5, and 7.
2. Calculate the sum of these odd numbers: 3 + 5 + 7 = 15.
3. Check if this sum (15) is divisible by 3: 15 ÷ 3 = 5, which is an integer.

Since we have at least two odd numbers (3 and 5, for example) whose combined sum (including the third odd number, 7, makes it even more obvious) results in a number divisible by 3, the sequence “3, 4, 5, 7, 10, 18, 22, 24” meets the criteria to be considered “Strange.”
